# Worldwide Earthquake Events API - Gold Layer Processing

In [1]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
# ensure the below library is installed on your fabric environment
import reverse_geocoder as rg

StatementMeta(, add6c82d-29cc-4e08-bdf7-07d5e96eaf97, 5, Finished, Available, Finished, False)

In [3]:
df = spark.read.table("earthquake_events_silver").filter(col('time') > start_date)

StatementMeta(, add6c82d-29cc-4e08-bdf7-07d5e96eaf97, 7, Finished, Available, Finished, False)

In [6]:
coordinates = (-122.515998840332,37.7036666870117)
rg.search(coordinates)[0].get('cc')

StatementMeta(, add6c82d-29cc-4e08-bdf7-07d5e96eaf97, 10, Finished, Available, Finished, False)

'TF'

In [7]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using the reverse geocoding API.

    Example:
    >>> get_country_details(48.8588443, 2.2943506)
    'FR'
    """
    coordinates = (float(lat), float(lon))
    return rg.search(coordinates)[0].get('cc')

StatementMeta(, add6c82d-29cc-4e08-bdf7-07d5e96eaf97, 11, Finished, Available, Finished, False)

In [8]:
# registering the udfs so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType())

StatementMeta(, add6c82d-29cc-4e08-bdf7-07d5e96eaf97, 12, Finished, Available, Finished, False)

In [9]:
# adding country_code and city attributes
df_with_location = \
                df.\
                    withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

StatementMeta(, add6c82d-29cc-4e08-bdf7-07d5e96eaf97, 13, Finished, Available, Finished, False)

In [10]:
# adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn('sig_class', 
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )

StatementMeta(, add6c82d-29cc-4e08-bdf7-07d5e96eaf97, 14, Finished, Available, Finished, False)

In [12]:
# appending the data to the gold table
df_with_location_sig_class.write.mode('append').saveAsTable('earthquake_events_gold')

StatementMeta(, add6c82d-29cc-4e08-bdf7-07d5e96eaf97, 16, Finished, Available, Finished, False)